[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/04-memoria-largo-plazo.ipynb)

# 04 — Memoria a largo plazo

En este notebook implementamos dos tipos de memoria persistente para agentes de IA:

- **Memoria episódica**: almacena hechos concretos en un fichero JSON. Rápida, sencilla, ideal para información estructurada.
- **Memoria semántica**: usa ChromaDB y embeddings para buscar información por similitud semántica. Potente, escalable, ideal para grandes volúmenes de texto.

Combinamos ambas en un `MemoryManager` que dota a un agente de Claude de memoria persistente entre conversaciones.

**Complemento del tutorial:** `tutoriales/agentes-avanzados/04-memoria-largo-plazo.md`

**Requisitos:** `ANTHROPIC_API_KEY` en las variables de entorno. ChromaDB y sentence-transformers se instalan en la siguiente celda.

In [ ]:
# %pip install chromadb sentence-transformers anthropic

import anthropic
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

cliente = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODELO = "claude-haiku-4-5-20251001"

print("Imports cargados. Iniciando módulos de memoria...")

## 1. Memoria episódica (JSON)

La memoria episódica almacena hechos discretos en un fichero JSON local. Es perfecta para guardar información estructurada como preferencias del usuario, datos importantes de sesiones anteriores o hechos específicos que el agente debe recordar.

In [ ]:
class MemoriaEpisodica:
    """
    Memoria episódica basada en fichero JSON.
    Almacena hechos discretos con timestamp y los recupera todos.
    """

    def __init__(self, ruta: str = "memoria_episodica.json"):
        self.ruta = Path(ruta)
        self._asegurar_fichero()

    def _asegurar_fichero(self) -> None:
        """Crea el fichero si no existe."""
        if not self.ruta.exists():
            self.ruta.write_text(json.dumps({"hechos": []}, ensure_ascii=False), encoding="utf-8")

    def _cargar(self) -> dict:
        """Carga el contenido del fichero JSON."""
        with open(self.ruta, "r", encoding="utf-8") as f:
            return json.load(f)

    def _guardar(self, datos: dict) -> None:
        """Persiste los datos en el fichero JSON."""
        with open(self.ruta, "w", encoding="utf-8") as f:
            json.dump(datos, f, ensure_ascii=False, indent=2)

    def guardar(self, hecho: str) -> None:
        """
        Guarda un hecho en la memoria episódica.

        Args:
            hecho: Texto del hecho a recordar (ej. "El usuario se llama Ana").
        """
        datos = self._cargar()
        datos["hechos"].append({
            "id": str(uuid.uuid4())[:8],
            "hecho": hecho,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        self._guardar(datos)

    def recordar(self) -> list[dict]:
        """
        Devuelve todos los hechos almacenados, ordenados del más reciente al más antiguo.

        Returns:
            Lista de diccionarios con los campos: id, hecho, timestamp.
        """
        datos = self._cargar()
        return list(reversed(datos["hechos"]))

    def vaciar(self) -> None:
        """Borra todos los hechos almacenados."""
        self._guardar({"hechos": []})


# --- Demostración ---
mem_epis = MemoriaEpisodica("demo_episodica.json")
mem_epis.vaciar()  # Empezar desde cero para la demo

# Guardar algunos hechos
mem_epis.guardar("El usuario se llama Carlos.")
mem_epis.guardar("Carlos trabaja como ingeniero de software.")
mem_epis.guardar("Carlos prefiere respuestas concisas y técnicas.")
mem_epis.guardar("Carlos ha mencionado que está aprendiendo sobre agentes de IA.")

# Recuperar todos los hechos
hechos = mem_epis.recordar()
print(f"Hechos almacenados: {len(hechos)}")
for h in hechos:
    print(f"  [{h['id']}] {h['hecho']}")

## 2. Memoria semántica (ChromaDB)

La memoria semántica usa **embeddings** (representaciones vectoriales del texto) para buscar información por similitud de significado, no por palabras clave exactas. ChromaDB es una base de datos vectorial ligera que funciona en memoria o en disco.

In [ ]:
class MemoriaSemantica:
    """
    Memoria semántica usando ChromaDB y sentence-transformers.
    Permite buscar recuerdos por similitud semántica.
    """

    def __init__(self, nombre_coleccion: str = "memoria"):
        try:
            import chromadb
            from sentence_transformers import SentenceTransformer

            self.cliente_db = chromadb.EphemeralClient()
            self.coleccion = self.cliente_db.create_collection(
                name=nombre_coleccion,
                metadata={"hnsw:space": "cosine"},
            )
            # Modelo ligero multilingüe para embeddings
            self.modelo_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            self._disponible = True
            print(f"MemoriaSemantica inicializada con coleccion '{nombre_coleccion}'.")
        except ImportError as e:
            self._disponible = False
            print(f"Dependencias no disponibles: {e}")
            print("Instala con: pip install chromadb sentence-transformers")

    def _embedding(self, texto: str) -> list[float]:
        """Genera el vector de embedding para un texto."""
        return self.modelo_embedding.encode(texto).tolist()

    def anadir(self, texto: str, metadatos: dict = None) -> str:
        """
        Añade un texto a la memoria semántica.

        Args:
            texto: El texto a almacenar.
            metadatos: Metadatos opcionales (ej. {"fuente": "conversacion", "turno": 3}).

        Returns:
            El ID asignado al documento.
        """
        if not self._disponible:
            return ""
        doc_id = str(uuid.uuid4())
        embedding = self._embedding(texto)
        self.coleccion.add(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[texto],
            metadatas=[metadatos or {"timestamp": datetime.now(timezone.utc).isoformat()}],
        )
        return doc_id

    def buscar(self, query: str, k: int = 3) -> list[dict]:
        """
        Busca los k fragmentos más similares semánticamente a la query.

        Args:
            query: Texto de búsqueda.
            k: Número de resultados a devolver.

        Returns:
            Lista de diccionarios con 'texto', 'distancia' y 'metadatos'.
        """
        if not self._disponible:
            return []
        total = self.coleccion.count()
        if total == 0:
            return []
        k_real = min(k, total)
        embedding_query = self._embedding(query)
        resultados = self.coleccion.query(
            query_embeddings=[embedding_query],
            n_results=k_real,
        )
        salida = []
        for i in range(len(resultados["ids"][0])):
            salida.append({
                "id": resultados["ids"][0][i],
                "texto": resultados["documents"][0][i],
                "distancia": round(resultados["distances"][0][i], 4),
                "metadatos": resultados["metadatas"][0][i],
            })
        return salida

    def total(self) -> int:
        """Devuelve el número de documentos almacenados."""
        if not self._disponible:
            return 0
        return self.coleccion.count()


# --- Demostración ---
mem_sem = MemoriaSemantica("demo_semantica")

if mem_sem._disponible:
    # Añadir fragmentos de texto
    mem_sem.anadir("Carlos trabaja como ingeniero de software en una empresa de fintech.")
    mem_sem.anadir("El proyecto principal de Carlos usa Python y FastAPI.")
    mem_sem.anadir("Carlos está interesado en aprender sobre machine learning y LLMs.")
    mem_sem.anadir("La empresa de Carlos tiene sede en Madrid y 50 empleados.")
    mem_sem.anadir("Carlos prefiere reuniones cortas y comunicación asíncrona.")

    print(f"Documentos almacenados: {mem_sem.total()}")

    # Buscar por similitud semántica
    query = "¿Qué tecnologías usa?"
    resultados = mem_sem.buscar(query, k=3)
    print(f"\nBúsqueda: '{query}'")
    for r in resultados:
        print(f"  [dist={r['distancia']}] {r['texto']}")

## 3. MemoryManager integrado

El `MemoryManager` combina la memoria episódica y la semántica en una interfaz única. Cuando el agente recibe un mensaje, primero recupera contexto relevante de la memoria semántica, añade los hechos episódicos más recientes y construye el contexto para la llamada a Claude.

In [ ]:
class MemoryManager:
    """
    Gestor de memoria que combina memoria episódica y semántica
    para dotar a un agente de Claude de contexto persistente.
    """

    def __init__(
        self,
        ruta_episodica: str = "memoria_episodica.json",
        nombre_coleccion: str = "agente_memoria",
        max_hechos_episodicos: int = 5,
        max_resultados_semanticos: int = 3,
    ):
        self.mem_episodica = MemoriaEpisodica(ruta_episodica)
        self.mem_semantica = MemoriaSemantica(nombre_coleccion)
        self.max_hechos = max_hechos_episodicos
        self.max_semanticos = max_resultados_semanticos
        self.historial = []  # Historial de la conversación actual

    def recordar_hecho(self, hecho: str) -> None:
        """
        Almacena un hecho en ambas memorias.
        Úsalo para persistir información importante de la conversación.
        """
        self.mem_episodica.guardar(hecho)
        self.mem_semantica.anadir(hecho, {"tipo": "hecho", "timestamp": datetime.now(timezone.utc).isoformat()})

    def _construir_contexto(self, mensaje_usuario: str) -> str:
        """Recupera y combina el contexto relevante de ambas memorias."""
        partes = []

        # Hechos episódicos más recientes
        hechos = self.mem_episodica.recordar()[:self.max_hechos]
        if hechos:
            hechos_texto = "\n".join(f"- {h['hecho']}" for h in hechos)
            partes.append(f"Hechos conocidos sobre el usuario:\n{hechos_texto}")

        # Recuerdos semánticamente relevantes
        if self.mem_semantica._disponible and self.mem_semantica.total() > 0:
            relevantes = self.mem_semantica.buscar(mensaje_usuario, k=self.max_semanticos)
            # Filtrar los que no estén ya en los hechos episódicos
            hechos_set = {h["hecho"] for h in hechos}
            relevantes_nuevos = [r for r in relevantes if r["texto"] not in hechos_set]
            if relevantes_nuevos:
                relevantes_texto = "\n".join(f"- {r['texto']}" for r in relevantes_nuevos)
                partes.append(f"Contexto relacionado con la consulta:\n{relevantes_texto}")

        return "\n\n".join(partes)

    def chat(self, mensaje: str) -> str:
        """
        Envía un mensaje al agente, que usa la memoria para enriquecer su respuesta.

        El agente:
          1. Recupera contexto relevante de la memoria.
          2. Construye el system prompt con ese contexto.
          3. Llama a Claude con el historial de la conversación.
          4. Guarda el intercambio en la memoria semántica.

        Args:
            mensaje: El mensaje del usuario.

        Returns:
            La respuesta del agente.
        """
        # Recuperar contexto relevante
        contexto = self._construir_contexto(mensaje)

        # Construir system prompt con memoria
        system = (
            "Eres un asistente personal con memoria persistente. "
            "Usas el contexto de conversaciones anteriores para dar respuestas personalizadas.\n\n"
        )
        if contexto:
            system += f"CONTEXTO DE MEMORIA:\n{contexto}\n\n"
        system += "Responde de forma concisa y personalizada según el contexto conocido."

        # Añadir mensaje al historial
        self.historial.append({"role": "user", "content": mensaje})

        # Llamar a Claude
        respuesta = cliente.messages.create(
            model=MODELO,
            max_tokens=400,
            system=system,
            messages=self.historial,
        )
        texto_respuesta = respuesta.content[0].text

        # Añadir respuesta al historial
        self.historial.append({"role": "assistant", "content": texto_respuesta})

        # Guardar el intercambio en la memoria semántica para futuras consultas
        self.mem_semantica.anadir(
            f"Usuario preguntó: {mensaje}",
            {"tipo": "conversacion", "turno": len(self.historial) // 2},
        )

        return texto_respuesta


print("MemoryManager definido.")

## 4. Demo

Simulamos una conversación de 4 turnos. Primero cargamos hechos en la memoria, luego preguntamos cosas que requieren recordar información de turnos anteriores.

In [ ]:
# Inicializar el MemoryManager con ficheros de demo
manager = MemoryManager(
    ruta_episodica="demo_manager_episodica.json",
    nombre_coleccion="demo_manager",
)

# Vaciar memorias para empezar desde cero
manager.mem_episodica.vaciar()

# Cargar hechos previos sobre el usuario (simulan sesiones anteriores)
print("Cargando hechos previos en la memoria...")
manager.recordar_hecho("El usuario se llama Elena.")
manager.recordar_hecho("Elena es diseñadora UX y trabaja en una startup de educación.")
manager.recordar_hecho("Elena está aprendiendo a programar en Python.")
manager.recordar_hecho("Su objetivo es automatizar tareas repetitivas con scripts.")
print("Hechos cargados.\n")

# --- Conversación de 4 turnos ---
turnos = [
    "Hola, ¿puedes recordarme qué estaba aprendiendo?",
    "¿Qué tipo de tareas podría automatizar con Python en mi trabajo?",
    "¿Por dónde me recomiendas empezar si soy diseñadora sin experiencia en código?",
    "¿Recuerdas cuál es mi objetivo principal?",
]

print("=" * 60)
print("CONVERSACIÓN CON MEMORIA PERSISTENTE")
print("=" * 60)

for i, mensaje in enumerate(turnos, 1):
    print(f"\n[Turno {i}]")
    print(f"Usuario: {mensaje}")
    respuesta = manager.chat(mensaje)
    print(f"Agente : {respuesta}")
    print("-" * 40)

print("\nConversación completada.")
print(f"Hechos en memoria episódica: {len(manager.mem_episodica.recordar())}")
if manager.mem_semantica._disponible:
    print(f"Documentos en memoria semántica: {manager.mem_semantica.total()}")